<a href="https://colab.research.google.com/github/Abdoulaye279/Financial-Engineering/blob/main/black-scholes-sp500/02_pricing_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Black-Scholes vs Monte Carlo and binomial tree

Checking the implementation: with the same assumptions (S&P 500 spot and vol), a Monte Carlo and a 2000-step CRR tree should give the BS price.

In [1]:
# colab: get the repo first
try:
    import google.colab
    !git clone -q https://github.com/Abdoulaye279/Financial-Engineering
    %cd Financial-Engineering/black-scholes-sp500
except ImportError:
    pass

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pricing import black_scholes, monte_carlo, binomial_crr, implied_vol
plt.style.use("quant.mplstyle")

df = pd.read_csv("data/sp500.csv")
df["Date"] = pd.to_datetime(df["Date"].str[:10])
prices = df.set_index("Date")["Close"].loc["2000":]
rets = np.log(prices).diff().dropna()

S0 = prices.iloc[-1]
sigma = rets.std() * np.sqrt(252)   # historical vol since 2000
r, T = 0.03, 21 / 252               # 3% rate, 1 month
print(len(rets), "days", rets.index[0].date(), "->", rets.index[-1].date(), "| S0", round(S0, 2), "| vol", round(sigma, 4))

6761 days 2000-01-04 -> 2026-11-19 | S0 7175.83 | vol 0.1924


In [3]:
print("1-month ATM call:", round(black_scholes(S0, S0, T, r, sigma), 2))
for m in [0.95, 1.00, 1.05]:
    for kind in ["call", "put"]:
        K = m * S0
        bs = black_scholes(S0, K, T, r, sigma, kind)
        mc, se = monte_carlo(S0, K, T, r, sigma, kind)
        crr = binomial_crr(S0, K, T, r, sigma, kind)
        print(f"K/S0={m}  {kind}  BS={bs:.2f}  MC={mc:.2f} [{mc-1.96*se:.2f}, {mc+1.96*se:.2f}]  CRR={crr:.2f}")
# the three agree within a few cents and BS is always inside the MC 95% interval -> code is right
# but MC and CRR use the same assumptions as BS, so this says nothing about the real market (see 03)

1-month ATM call: 167.92
K/S0=0.95  call  BS=410.12  MC=410.23 [409.55, 410.91]  CRR=410.13
K/S0=0.95  put  BS=34.31  MC=34.39 [34.19, 34.59]  CRR=34.32
K/S0=1.0  call  BS=167.92  MC=168.16 [167.67, 168.64]  CRR=167.90


K/S0=1.0  put  BS=150.00  MC=150.21 [149.78, 150.64]  CRR=149.98
K/S0=1.05  call  BS=46.21  MC=46.32 [46.06, 46.58]  CRR=46.22
K/S0=1.05  put  BS=386.19  MC=386.27 [385.63, 386.91]  CRR=386.20
